# 02. Pipeline walkthrough (evaluator view)

**LAST TRAY: Flavoria DiningOps Truth.** A concise, read-only tour of the already-built pipeline. Every value below is loaded from the committed files in `outputs/`; nothing is recomputed, and no raw data is read.

## 1. Project question

Can we reconstruct a trustworthy operational view of dining measurements from the available source data, and determine whether that evidence is sufficient to support future food-waste decisions?

This is an FDE-style reconstruction using publicly available Flavoria research data and public FMI weather data. It is not an analysis of Flavoria's operational systems.

In [1]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)

# find the repository root from wherever the notebook is opened
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "outputs" / "metrics" / "metrics.csv").exists())
OUT = ROOT / "outputs"


def csv(rel):
    return pd.read_csv(OUT / rel, dtype=str, keep_default_na=False)


def js(rel):
    return json.loads((OUT / rel).read_text(encoding="utf-8"))


print("Reading committed pipeline outputs from:", OUT.relative_to(ROOT))

Reading committed pipeline outputs from: outputs


## 2. Source map summary

| Source | Class | Role |
|---|---|---|
| FlavoriaFoodWeight1700 (Zenodo) | primary measurement data | one component weighing event per row |
| FMI open data weather API | weather enrichment | hourly context only |
| Flavoria Data Catalog | contextual documentation | definitions and access status |
| Weigh & Dine documentation | source-gap documentation | describes a checkout total; no sample |
| Lunch Line Waste documentation | source-gap documentation | describes per-tray waste; not publicly accessible |

The table below is the ingestion manifest: what was retrieved and whether each artifact verified against its pinned size, checksum and row count.

In [2]:
manifest = csv("ingestion/raw_artifact_manifest.csv")
manifest.groupby(["source_name", "lane", "kind", "status"]).size().rename("artifacts").reset_index()

,source_name,lane,kind,status,artifacts
0,flavoria,core,archive,VERIFIED,1
1,flavoria,core,member,VERIFIED,11
2,fmi_weather,context,evidence,VERIFIED,2
3,fmi_weather,context,weather_chunk,VERIFIED,7


## 3. Pipeline stages

Six gated stages run in order; a stage runs only if the one before it passed, and each re-verifies its upstream inputs. Below: the committed stage summary and the pipeline controls.

In [3]:
stages = csv("pipeline/stage_summary.csv")[["stage", "status", "gate", "warnings", "errors", "output_files"]]
controls = csv("pipeline/pipeline_controls.csv")[["control_id", "control", "status"]]
display_stages = stages
print(controls.status.value_counts().to_dict(), "(pipeline controls by status)")
display_stages

{'PASS': 9, 'INFO': 2} (pipeline controls by status)


,stage,status,gate,warnings,errors,output_files
0,ingest,PASSED,open,0,0,6
1,stage,PASSED,ingest PASSED,0,0,4
2,validate,PASSED,stage PASSED,530,4,9
3,model,PASSED,validate PASSED,0,0,8
4,metrics,PASSED,model PASSED,0,0,6
5,sensitivity,PASSED,metrics PASSED,0,0,13


## 4. Validation summary

Findings are classified by rule and severity. Quarantine is a flag plus a manifest: records are kept and listed, never deleted.

In [4]:
v = js("validation/validation_summary.json")
pd.DataFrame({"value": {
    "core status": v["core_status"],
    "findings by severity": v["issues"]["by_severity"],
    "reconciliation (checks / pass / warn / fail / info)": " / ".join(str(v["reconciliation"][k]) for k in ("checks", "pass", "warn", "fail", "info")),
    "quarantined session keys": v["quarantine"]["session_keys"],
    "quarantined events": v["quarantine"]["events"],
    "quarantined session ids": ", ".join(v["quarantine"]["session_ids"]),
}})

,value
core status,PASSED_WITH_QUARANTINE
findings by severity,"{'ERROR': 4, 'INFO': 849, 'WARN': 530}"
reconciliation (checks / pass / warn / fail / info),46 / 35 / 0 / 0 / 11
quarantined session keys,4
quarantined events,22
quarantined session ids,"session2266, session3222"


## 5. Canonical business model summary

Grain first: the atomic row is a weighing event; the session is derived; the session key is `(session_id, population)`. Below are the committed model tables with their grain, and the model's own control checks.

In [5]:
m = js("model/model_manifest.json")
tables = pd.DataFrame([{"table": name, "rows": t["rows"], "grain": t["grain"]} for name, t in m["tables"].items()])
print("model controls:", {k: m["controls"][k] for k in ("checks", "pass", "warn", "fail", "info")})
tables

model controls: {'checks': 26, 'pass': 19, 'warn': 0, 'fail': 0, 'info': 7}


,table,rows,grain
0,fact_daily_volume,65,one service date x population
1,fact_dining_session,3345,"one DERIVED session key (session_id, population)"
2,fact_session_component,11925,one distinct normalised component within one session key (MODELLABLE events only)
3,fact_weather,1129,one FMI station x UTC hour observation (one column per requested parameter)
4,fact_weighing_event,12284,one observed component weighing event (one source data row)


## 6. Final evidence: M1-M5 (with S2 and W1)

Values are read from `outputs/metrics/metrics.csv`. M1-M4 describe the core-ready registered-export sessions; M5 and S2 use the fixed eligible denominator.

![Derived selected meal weight, M1 and M2](../diagrams/weight-distribution.png)

In [6]:
metrics = csv("metrics/metrics.csv")
wanted = ["M1", "M2", "M3", "M4", "M5", "S2", "W1"]
metrics[metrics.metric_id.isin(wanted)][["metric_id", "metric_name", "value_display", "population"]].reset_index(drop=True)

,metric_id,metric_name,value_display,population
0,M1,Median Derived Selected Meal Weight,499 g,core_ready_registered_export_sessions
1,M2,P90 Derived Selected Meal Weight,"1,039.6 g",core_ready_registered_export_sessions
2,M3,Observed Valid Sessions — Registered-Export Population,"1,697 sessions",core_ready_registered_export_sessions
3,M4,Median Distinct Normalized Components per Session,5 components,core_ready_registered_export_sessions
4,M5,Core Measurement Readiness,99.88%,eligible_registered_export_sessions
5,S2,Warn-Free Rate,97.88%,eligible_registered_export_sessions
6,W1,Direct Food Waste Measurement,BLOCKED / SOURCE GAP,n/a


## 7. Sensitivity summary

The baseline is frozen; each scenario changes one documented assumption. Scenarios are sensitivity tests, not alternative truths.

![M1 and M2 across scenarios](../outputs/evidence/figures/m1_m2_sensitivity.png)

In [7]:
s = js("evidence/sensitivity_summary.json")
print("scenarios registered:", s["scenarios"]["registered"], "| conclusion classes:", s["conclusion_classes"])
print("ranges:", {k: (s["ranges"][k]["min"], s["ranges"][k]["max"]) for k in ("M1", "M2")})
matrix = csv("evidence/evidence_matrix.csv")
matrix[["Question", "Robustness", "Observed range/change"]]

scenarios registered: 26 | conclusion classes: {'BLOCKED': 2, 'CONDITIONAL': 2, 'SENSITIVE': 3, 'STABLE': 6}
ranges: {'M1': (493.0, 505.0), 'M2': (977.0, 1066.0)}


,Question,Robustness,Observed range/change
0,Can a selected-meal-weight distribution be reconstructed from the available component weighing e...,STABLE,"M1: S04 +0.00, S05 +0.00, S33 +0.00; M2: S04 -1.60, S05 -1.60, S33 +0.00"
1,Is the median derived selected meal weight of the same order of magnitude under reasonable scena...,STABLE,M1 499.0 -> 493.0 to 505.0
2,How stable is the upper end of the distribution (P90)?,SENSITIVE,"M2 1,039.6 -> 977.0 to 1,066.0"
3,Does the component-count result remain stable?,STABLE,M4 5.0 -> 5.0 to 5.0
4,Is the registered-export valid-session count and readiness stable to the timezone and crossover ...,CONDITIONAL,"M3: S01 +2.00, S02 +1.00, S03 +1.00, TZ0 -303.00, TZ1 -103.00, TZ2 +0.00, TZ3 +0.00, TZ4 +0.00; ..."
5,Does the treatment of the two crossover sessions change any conclusion?,STABLE,"M1: S01 +0.00, S02 +0.00, S03 +0.50; M2: S01 +2.80, S02 -0.40, S03 +3.00; M4: S01 +0.00, S02 +0...."
6,Do the six irregular-volume days change the measurement profile?,SENSITIVE,"M1: S20 +6.00, S21 +0.50, S22 -6.00; M2: S20 +26.40, S21 +12.50, S22 +13.40; M4: S20 +0.00, S21 ..."
7,Does the study period matter (the first two weeks)?,SENSITIVE,M1: S23 +6.00; M2: S23 -62.60; M4: S23 +0.00
8,"Is the 2,097 g observation influential?",STABLE,"M1: S04 +0.00, S05 +0.00; M2: S04 -1.60, S05 -1.60; M4: S04 +0.00, S05 +0.00"
9,Should the warn-free rate be read as a second readiness score?,STABLE,M1: S10 +0.50; M2: S10 -24.60


## 8. Known vs unknown

**Known:** event-level weight exists; sessions can be reconstructed; a selected meal weight can be derived; weather aligns for the core-ready sessions; two crossover session ids exist.
**Unknown:** how much was consumed; what was left over; actual food waste.

Assumptions and their measured sensitivity, from the committed uncertainty register:

In [8]:
register = csv("evidence/uncertainty_register.csv")
register[["uncertainty_id", "assumption", "impact_level", "sensitivity_result"]]

,uncertainty_id,assumption,impact_level,sensitivity_result
0,U01,"Suspect export timezone: the file's timestamps are 3 hours early (a +3h, file-specific normaliza...",HIGH,+2h/+3h/+4h give identical M1-M5; +0h quarantines 303 sessions (M5 82.05%); +1h quarantines 103 ...
1,U02,Population labels (registered-export / non-registered-export) come from file names and mean only...,HIGH,"The diagnostic non-registered-export profile is M1 192 g, M2 855.4 g, M4 2; the forbidden pooled..."
2,U03,Crossover sessions: two session ids present in both exports are quarantined rather than resolved.,LOW,Including both changes M2 by +2.8 g and M5 to 100.0% only because the check is removed.
3,U04,"Component identity is the trimmed, whitespace-collapsed, case-folded name; there is no alias table.",LOW,"M4 is 5 with raw names, 5 counting scales and 5 counting weighing events."
4,U05,Six days whose observed volume regime differs from the weekday baseline are real observations.,MEDIUM,"Excluding them removes 252 sessions and moves M2 by +26.4 g (+2.5%), M1 by +6 g."
5,U06,"Derived selected meal weight is what was weighed at the line, not what was eaten.",BLOCKING,Not testable: no scenario can produce consumption from selection.
6,U07,Direct food-waste measurement is unavailable.,BLOCKING,"Not testable: no lower or upper waste bound, band or proxy is produced."
7,U08,"The Turku weather station is a regional context, and r_1h is the hour ending at its timestamp.",LOW,"Under the timezone hypotheses, all 385 sessions in the shifted file join a different weather hou..."
8,U09,Five weeks of data represent the measurement profile.,MEDIUM,Excluding them moves M2 by -62.6 g (-6.0%); M1 by +6 g.
9,U10,"The diagnostic thresholds (B02, B07, T05, C02) describe this data and are not physical limits.",LOW,Applying B07 as an exclusion moves M2 by -19.1 g; any WARN as an exclusion by -24.6 g.


## 9. Selected weight is not consumption, and consumption is not waste

The evidence chain the pipeline maintains, and the one conclusion class that no scenario can change.

In [9]:
summary = js("metrics/metric_summary.json")
print("  ->  ".join(summary["semantic_chain"]))
print()
print(summary["waste"])
matrix[matrix.Robustness == "BLOCKED"][["Question", "Robustness", "What cannot be concluded"]]

OBSERVED component weighing events  ->  DERIVED selected meal weight  ->  UNKNOWN actual consumed quantity  ->  SOURCE GAP actual food waste

W1 BLOCKED / SOURCE GAP: no direct waste records are available; nothing is estimated from any weight or other proxy


,Question,Robustness,What cannot be concluded
11,Can actual consumption be determined from selected meal weight alone?,BLOCKED,"Any amount consumed, left over or eaten."
12,Can food waste be measured from the accessible data?,BLOCKED,"Any waste amount, potential waste, waste band or ratio."


## 10. Where to look next

| Topic | Document | Source module |
|---|---|---|
| Sources and truth decisions | `docs/source_map.md`, `docs/source_truth_decisions.md` | `src/ingest/`, `config/sources.yml` |
| Validation rules | `docs/validation_rules.md` | `src/validate/` |
| Canonical model | `docs/data_dictionary.md` (section 12) | `src/model/` |
| Metrics | `docs/metric_contract.md`, `docs/final_evidence.md` | `src/metrics/` |
| Sensitivity | `docs/sensitivity_analysis.md` | `src/sensitivity/` |
| Pipeline | `docs/pipeline.md` | `src/pipeline/run.py` |
| Assumptions and limits | `docs/known_unknowns_assumptions_limitations.md` | |
| The FDE judgement | `docs/judgement_call.md` | |

> This notebook is an evaluator walkthrough only; the canonical computations live in `src/` and are executed by `python -m src.pipeline.run --stages all`.